# Track3 End-to-End Learner Notebook

이 노트북 **하나만으로** Track3 실습의 전체 절차를 처음부터 끝까지 순서대로 실행해볼 수 있습니다.

기존 [Track3_Mission_Workbench.ipynb](./Track3_Mission_Workbench.ipynb)는 `subprocess`로 스크립트를 호출하지만,
이 노트북은 아래 3개 스크립트의 **로직 자체를 셀로 옮겨와** 각 단계를 직접 읽고, 실행하고, 값을 바꿔보며 학습할 수 있도록 만들었습니다.

- `generate_track3_samples.py` → **1부. 샘플 데이터 생성**
- `run_track3_simulation.py` → **2부. 통합 응답 시뮬레이션 (정상 + fallback)**
- `evaluate_track3_outputs.py` → **3부. 품질 게이트 평가**

## 실습 흐름 (고정 정책)

1. **킥오프**: Track1 정형 데이터 + Track2 비정형 매니페스트를 조합해 Q1~Q3 시나리오 입력 생성
2. **통합응답**: Tool A(FabricIQ, 정형) / Tool B(WorkIQ, 비정형) 결과를 합쳐 답변 구성
3. **fallback**: 도구 실패 시 **5초 → 10초 → 20초 재시도(최대 3회)** 후 부분/차단 응답 정책 적용
4. **평가**: 8개 품질 게이트 항목 기준으로 pass/fail 판정 (`evaluate_track3_outputs.py`와 동일 로직)

> 참고 문서: [track3/WORKBOOK.md](../WORKBOOK.md), [track3/PREREQUISITES.md](../PREREQUISITES.md), [track3/data/README.md](./README.md)

각 섹션은 순서대로 실행하세요 (Run All 가능). 셀 안의 함수/값을 직접 바꿔서 실행 결과가 어떻게 달라지는지 실험해보는 것을 권장합니다.


## 0. 환경 설정

저장소 루트 또는 `track3/data` 폴더 어디에서 열어도 동작하도록 경로를 자동으로 찾습니다.

In [1]:
from __future__ import annotations

import csv
import json
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if (ROOT / "track3" / "data").is_dir():
    TRACK3_ROOT = ROOT / "track3" / "data"
elif ROOT.name == "data" and ROOT.parent.name == "track3":
    TRACK3_ROOT = ROOT
else:
    raise RuntimeError("track3/data 폴더를 찾을 수 없습니다. 저장소 루트 또는 track3/data 폴더에서 실행하세요.")

REPO_ROOT = TRACK3_ROOT.parent.parent
TRACK1_ROOT = REPO_ROOT / "track1" / "data"
TRACK2_MANIFEST = REPO_ROOT / "track2" / "data" / "generated" / "manifests" / "content_manifest.csv"
OUTPUT_DIR = TRACK3_ROOT / "generated"
RESPONSE_DIR = OUTPUT_DIR / "responses"
REPORT_PATH = OUTPUT_DIR / "reports" / "evaluation_report.json"
MARKDOWN_REPORT_PATH = OUTPUT_DIR / "reports" / "evaluation_report.md"

print("TRACK3_ROOT     =", TRACK3_ROOT)
print("TRACK1_ROOT     =", TRACK1_ROOT)
print("TRACK2_MANIFEST =", TRACK2_MANIFEST)
print("OUTPUT_DIR      =", OUTPUT_DIR)


TRACK3_ROOT     = /Users/hyungilkim/Documents/Data Platform Workshop/track3/data
TRACK1_ROOT     = /Users/hyungilkim/Documents/Data Platform Workshop/track1/data
TRACK2_MANIFEST = /Users/hyungilkim/Documents/Data Platform Workshop/track2/data/generated/manifests/content_manifest.csv
OUTPUT_DIR      = /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated


---
# 1부. 샘플 데이터 생성 (`generate_track3_samples.py` 로직)

Track1 CSV(정형 데이터)와 Track2 콘텐츠 매니페스트(비정형 문서 목록)를 읽어 Q1~Q3 시나리오의
Tool A 지표(`tool_a_metrics.json`), Tool B 근거(`tool_b_evidence.json`)를 생성합니다.


### 1-1. 고정 시나리오 정의 (Q1~Q3)

In [2]:
CORE_CAMPAIGNS = {"SummerPush", "BackToSchool", "VIPRetention", "FlashWeek"}
CORE_PRODUCTS = {"AeroPhone X", "SmartWatch Pro", "UltraBook 15"}
SUCCESS_PAYMENT_STATUSES = {"success", "retrysuccess"}

SCENARIOS: list[dict[str, Any]] = [
    {
        "id": "Q1",
        "question": "결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?",
        "goal": "캠페인별 전환율과 결제 실패 패턴 비교",
        "keywords": ["SummerPush", "VIPRetention", "결제", "payment", "전환율"],
        "semanticKeys": ["CampaignId", "OrderId", "PaymentStatus"],
    },
    {
        "id": "Q2",
        "question": "배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?",
        "goal": "배송 지연군의 반품/불만율 확인",
        "keywords": ["배송", "LateDelivery", "logistics", "return", "ticket"],
        "semanticKeys": ["OrderId", "DeliveryStatus", "ReturnId", "TicketId"],
    },
    {
        "id": "Q3",
        "question": "Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?",
        "goal": "AeroPhone X/SmartWatch Pro/UltraBook 15 성과 비교",
        "keywords": ["AeroPhone X", "SmartWatch Pro", "UltraBook 15", "Q3", "제품"],
        "semanticKeys": ["ProductId", "OrderId", "ReturnId"],
    },
]

for scenario in SCENARIOS:
    print(f"{scenario['id']}: {scenario['question']}")


Q1: 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?
Q2: 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?
Q3: Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?


### 1-2. 공용 유틸 함수 (CSV 읽기, 형변환, 텍스트 정규화)

In [3]:
@dataclass
class EvidenceItem:
    item_id: str
    source: str
    title: str
    business_date: str
    owner: str
    location: str
    target: str

    def to_dict(self) -> dict[str, str]:
        return {
            "id": self.item_id,
            "source": self.source,
            "title": self.title,
            "businessDate": self.business_date,
            "owner": self.owner,
            "location": self.location,
            "target": self.target,
        }


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    if not path.is_file():
        raise RuntimeError(f"CSV file not found: {path}")
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def to_float(value: str) -> float:
    if value is None:
        return 0.0
    token = value.strip()
    if not token:
        return 0.0
    return float(token)


def normalize_text(value: str) -> str:
    return " ".join(value.strip().split())


def write_json(path: Path, payload: Any, *, pretty: bool = True) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if pretty:
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    else:
        path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

print("유틸 함수 정의 완료")


유틸 함수 정의 완료


### 1-3. Q1 지표 — 결제 실패가 캠페인 전환율에 미치는 영향

In [4]:
def build_q1_metrics(track1_root: Path) -> dict[str, Any]:
    campaigns = read_csv_rows(track1_root / "campaigns.csv")
    campaign_names = {row["campaign_id"]: row["campaign_name"] for row in campaigns if row["campaign_name"] in CORE_CAMPAIGNS}

    attribution = read_csv_rows(track1_root / "campaign_attribution.csv")
    payments = read_csv_rows(track1_root / "payments.csv")

    payment_status_by_order: dict[str, str] = {}
    for row in payments:
        order_id = normalize_text(row.get("order_id", ""))
        if not order_id:
            continue
        status = normalize_text(row.get("payment_status", ""))
        if status:
            payment_status_by_order[order_id] = status

    orders_by_campaign: dict[str, set[str]] = defaultdict(set)
    for campaign_id in campaign_names:
        orders_by_campaign[campaign_id]
    for row in attribution:
        campaign_id = normalize_text(row.get("campaign_id", ""))
        order_id = normalize_text(row.get("order_id", ""))
        if campaign_id in campaign_names and order_id:
            orders_by_campaign[campaign_id].add(order_id)

    per_campaign: list[dict[str, Any]] = []
    for campaign_id, campaign_name in sorted(campaign_names.items(), key=lambda item: item[1]):
        order_ids = orders_by_campaign[campaign_id]
        success_count = 0
        failed_count = 0
        for order_id in order_ids:
            status = payment_status_by_order.get(order_id, "").lower()
            if status in SUCCESS_PAYMENT_STATUSES:
                success_count += 1
            else:
                failed_count += 1
        total = len(order_ids)
        conversion_rate = round((success_count / total) * 100, 2) if total else 0.0
        per_campaign.append(
            {
                "campaignId": campaign_id,
                "campaignName": campaign_name,
                "orders": total,
                "paymentSuccessOrders": success_count,
                "paymentFailedOrUnknownOrders": failed_count,
                "conversionRatePct": conversion_rate,
            }
        )

    sorted_rates = sorted(per_campaign, key=lambda row: row["conversionRatePct"], reverse=True)
    best = sorted_rates[0]["campaignName"] if sorted_rates else "-"
    worst = sorted_rates[-1]["campaignName"] if sorted_rates else "-"

    highlights = [
        f"핵심 캠페인 {len(per_campaign)}개를 비교했고 최고 전환율은 {best}, 최저 전환율은 {worst}이다.",
        "payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.",
    ]

    return {
        "scenarioId": "Q1",
        "title": "결제 실패-전환율 영향 분석",
        "highlights": highlights,
        "perCampaign": per_campaign,
    }

q1_metrics = build_q1_metrics(TRACK1_ROOT)
q1_metrics


{'scenarioId': 'Q1',
 'title': '결제 실패-전환율 영향 분석',
 'highlights': ['핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.',
  'payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.'],
 'perCampaign': [{'campaignId': 'CA00002',
   'campaignName': 'BackToSchool',
   'orders': 5,
   'paymentSuccessOrders': 3,
   'paymentFailedOrUnknownOrders': 2,
   'conversionRatePct': 60.0},
  {'campaignId': 'CA00004',
   'campaignName': 'FlashWeek',
   'orders': 4,
   'paymentSuccessOrders': 2,
   'paymentFailedOrUnknownOrders': 2,
   'conversionRatePct': 50.0},
  {'campaignId': 'CA00001',
   'campaignName': 'SummerPush',
   'orders': 0,
   'paymentSuccessOrders': 0,
   'paymentFailedOrUnknownOrders': 0,
   'conversionRatePct': 0.0},
  {'campaignId': 'CA00003',
   'campaignName': 'VIPRetention',
   'orders': 0,
   'paymentSuccessOrders': 0,
   'paymentFailedOrUnknownOrders': 0,
   'conversionRatePct': 0.0}]}

### 1-4. Q2 지표 — 배송 지연이 반품률/불만 티켓에 미치는 영향

In [5]:
def build_q2_metrics(track1_root: Path) -> dict[str, Any]:
    shipments = read_csv_rows(track1_root / "shipments.csv")
    returns = read_csv_rows(track1_root / "returns.csv")
    tickets = read_csv_rows(track1_root / "support_tickets.csv")

    delayed_orders: set[str] = set()
    for row in shipments:
        status = normalize_text(row.get("shipment_status", "")).lower()
        order_id = normalize_text(row.get("order_id", ""))
        if order_id and "delay" in status:
            delayed_orders.add(order_id)

    returned_orders: set[str] = set()
    delayed_return_reasons: Counter[str] = Counter()
    for row in returns:
        order_id = normalize_text(row.get("order_id", ""))
        reason = normalize_text(row.get("return_reason", "")) or "Unknown"
        if not order_id:
            continue
        returned_orders.add(order_id)
        if order_id in delayed_orders:
            delayed_return_reasons[reason] += 1

    complaint_orders: set[str] = set()
    for row in tickets:
        order_id = normalize_text(row.get("order_id", ""))
        ticket_type = normalize_text(row.get("ticket_type", "")).upper()
        if order_id and ticket_type == "COMPLAINT":
            complaint_orders.add(order_id)

    delayed_return_orders = delayed_orders & returned_orders
    delayed_complaint_orders = delayed_orders & complaint_orders
    delayed_count = len(delayed_orders)

    delayed_return_rate = round((len(delayed_return_orders) / delayed_count) * 100, 2) if delayed_count else 0.0
    delayed_complaint_rate = round((len(delayed_complaint_orders) / delayed_count) * 100, 2) if delayed_count else 0.0

    top_reasons = [{"reason": reason, "count": count} for reason, count in delayed_return_reasons.most_common(3)]

    highlights = [
        f"배송 지연 주문 {delayed_count}건 중 반품 발생 비율은 {delayed_return_rate}%이다.",
        f"배송 지연 주문의 COMPLAINT 티켓 비율은 {delayed_complaint_rate}%이다.",
    ]

    return {
        "scenarioId": "Q2",
        "title": "배송 지연 영향 분석",
        "highlights": highlights,
        "delayedOrderCount": delayed_count,
        "delayedReturnRatePct": delayed_return_rate,
        "delayedComplaintRatePct": delayed_complaint_rate,
        "topDelayedReturnReasons": top_reasons,
    }

q2_metrics = build_q2_metrics(TRACK1_ROOT)
q2_metrics


{'scenarioId': 'Q2',
 'title': '배송 지연 영향 분석',
 'highlights': ['배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.',
  '배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.'],
 'delayedOrderCount': 669,
 'delayedReturnRatePct': 49.78,
 'delayedComplaintRatePct': 16.14,
 'topDelayedReturnReasons': [{'reason': 'LateDelivery', 'count': 67},
  {'reason': 'Damaged', 'count': 67},
  {'reason': 'NotAsDescribed', 'count': 67}]}

### 1-5. Q3 지표 — 핵심 상품 3종 성과 비교

In [6]:
def build_q3_metrics(track1_root: Path) -> dict[str, Any]:
    products = read_csv_rows(track1_root / "products.csv")
    order_items = read_csv_rows(track1_root / "order_items.csv")
    returns = read_csv_rows(track1_root / "returns.csv")

    product_name_by_id = {row["product_id"]: row["product_name"] for row in products}
    target_product_ids = {product_id for product_id, name in product_name_by_id.items() if name in CORE_PRODUCTS}

    summary: dict[str, dict[str, Any]] = {}
    for product_name in sorted(CORE_PRODUCTS):
        summary[product_name] = {"orderIds": set(), "units": 0.0, "salesAmount": 0.0, "returns": 0}

    for row in order_items:
        product_id = normalize_text(row.get("product_id", ""))
        order_id = normalize_text(row.get("order_id", ""))
        if product_id not in target_product_ids or not order_id:
            continue
        product_name = product_name_by_id[product_id]
        summary_row = summary[product_name]
        summary_row["orderIds"].add(order_id)
        summary_row["units"] += to_float(row.get("quantity", "0"))
        summary_row["salesAmount"] += to_float(row.get("sales_amount", "0"))

    for row in returns:
        product_id = normalize_text(row.get("product_id", ""))
        if product_id in target_product_ids:
            product_name = product_name_by_id[product_id]
            summary[product_name]["returns"] += 1

    per_product: list[dict[str, Any]] = []
    for product_name in sorted(summary):
        row = summary[product_name]
        order_count = len(row["orderIds"])
        per_product.append(
            {
                "productName": product_name,
                "orderCount": order_count,
                "units": int(row["units"]),
                "salesAmount": round(row["salesAmount"], 2),
                "returnCount": row["returns"],
                "returnRatePct": round((row["returns"] / order_count) * 100, 2) if order_count else 0.0,
            }
        )

    highlights = [
        "Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.",
        "주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.",
    ]

    return {
        "scenarioId": "Q3",
        "title": "핵심 상품 3종 성과 비교",
        "highlights": highlights,
        "perProduct": per_product,
    }

q3_metrics = build_q3_metrics(TRACK1_ROOT)
q3_metrics


{'scenarioId': 'Q3',
 'title': '핵심 상품 3종 성과 비교',
 'highlights': ['Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.',
  '주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.'],
 'perProduct': [{'productName': 'AeroPhone X',
   'orderCount': 4,
   'units': 11,
   'salesAmount': 313.5,
   'returnCount': 5,
   'returnRatePct': 125.0},
  {'productName': 'SmartWatch Pro',
   'orderCount': 3,
   'units': 3,
   'salesAmount': 96.6,
   'returnCount': 0,
   'returnRatePct': 0.0},
  {'productName': 'UltraBook 15',
   'orderCount': 4,
   'units': 7,
   'salesAmount': 95.9,
   'returnCount': 0,
   'returnRatePct': 0.0}]}

### 1-6. Tool B 근거(비정형 문서) 선정 — Track2 매니페스트에서 키워드 매칭

In [7]:
def score_manifest_row(row: dict[str, str], keywords: list[str]) -> int:
    title = normalize_text(row.get("title", "")).lower()
    keyword_blob = normalize_text(row.get("keywords", "")).lower()
    score = 0
    for keyword in keywords:
        token = keyword.lower()
        if token in keyword_blob:
            score += 3
        if token in title:
            score += 2
    return score


def pick_evidence(manifest_rows: list[dict[str, str]], keywords: list[str], limit: int = 8) -> list[EvidenceItem]:
    candidates: list[tuple[int, EvidenceItem]] = []
    for row in manifest_rows:
        score = score_manifest_row(row, keywords)
        if score <= 0:
            continue
        candidates.append(
            (
                score,
                EvidenceItem(
                    item_id=normalize_text(row.get("id", "")),
                    source=normalize_text(row.get("source", "")),
                    title=normalize_text(row.get("title", "")),
                    business_date=normalize_text(row.get("businessDate", "")),
                    owner=normalize_text(row.get("owner", "")),
                    location=normalize_text(row.get("location", "")),
                    target=normalize_text(row.get("target", "")),
                ),
            )
        )

    candidates.sort(key=lambda item: (item[0], item[1].business_date), reverse=True)

    selected: list[EvidenceItem] = []
    seen_sources: set[str] = set()
    for _, item in candidates:
        if item.source and item.source not in seen_sources:
            selected.append(item)
            seen_sources.add(item.source)
            if len(selected) >= limit:
                return selected

    for _, item in candidates:
        if item in selected:
            continue
        selected.append(item)
        if len(selected) >= limit:
            break
    return selected

manifest_rows = read_csv_rows(TRACK2_MANIFEST)
source_counts = Counter(row.get("source", "").strip() for row in manifest_rows if row.get("source"))
print(f"manifestItems={len(manifest_rows)}, manifestSources={dict(source_counts)}")


manifestItems=60, manifestSources={'SharePoint': 15, 'OneDrive': 12, 'Outlook': 15, 'Teams': 18}


### 1-7. 시나리오별 산출물 조립 및 저장 (`scenarios.json`, `tool_a_metrics.json`, `tool_b_evidence.json`, `track3_seed_summary.json`)

In [8]:
tool_a_metrics: dict[str, Any] = {"Q1": q1_metrics, "Q2": q2_metrics, "Q3": q3_metrics}

tool_b_evidence: dict[str, dict[str, Any]] = {}
for scenario in SCENARIOS:
    evidence = pick_evidence(manifest_rows, scenario["keywords"])
    tool_b_evidence[scenario["id"]] = {
        "scenarioId": scenario["id"],
        "keywords": scenario["keywords"],
        "evidence": [item.to_dict() for item in evidence],
        "sourceCoverage": dict(Counter(item.source for item in evidence)),
    }

scenario_payload = {
    "generatedFrom": {
        "track1Root": str(TRACK1_ROOT),
        "track2Manifest": str(TRACK2_MANIFEST),
    },
    "scenarios": SCENARIOS,
}
summary_payload = {
    "manifestTotalItems": len(manifest_rows),
    "manifestSourceCounts": dict(source_counts),
    "scenarioCount": len(SCENARIOS),
}

write_json(OUTPUT_DIR / "scenarios.json", scenario_payload, pretty=True)
write_json(OUTPUT_DIR / "tool_a_metrics.json", tool_a_metrics, pretty=True)
write_json(OUTPUT_DIR / "tool_b_evidence.json", tool_b_evidence, pretty=True)
write_json(OUTPUT_DIR / "track3_seed_summary.json", summary_payload, pretty=True)

print("[Track3 Sample Generation]")
print(f"- outputDir: {OUTPUT_DIR}")
print(f"- scenarios: {len(SCENARIOS)}")
print(f"- manifestItems: {len(manifest_rows)}")
print(f"- manifestSources: {dict(source_counts)}")


[Track3 Sample Generation]
- outputDir: /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated
- scenarios: 3
- manifestItems: 60
- manifestSources: {'SharePoint': 15, 'OneDrive': 12, 'Outlook': 15, 'Teams': 18}


---
# 2부. 통합 응답 시뮬레이션 (`run_track3_simulation.py` 로직)

Tool A/Tool B를 호출하고, 실패 모드(`mode`)에 따라 **5초 → 10초 → 20초 재시도(최대 3회)** 정책을 적용한 뒤
통합 응답(`overallStatus`, `warnings`, `evidenceLinks` 등)을 구성합니다.


### 2-1. 실패 모드 정의 및 재시도 정책

In [9]:
MODES = {
    "normal",
    "tool-a-down",
    "tool-b-down",
    "both-down",
    "tool-a-transient",
    "tool-b-transient",
}


def parse_retry_delays(value: str, max_retries: int) -> list[int]:
    items = [chunk.strip() for chunk in value.split(",") if chunk.strip()]
    delays = [int(chunk) for chunk in items]
    if len(delays) < max_retries:
        delays.extend([delays[-1] if delays else 0] * (max_retries - len(delays)))
    return delays[:max_retries]


def should_tool_fail(mode: str, tool_name: str, attempt: int) -> bool:
    if mode == "both-down":
        return True
    if mode == "tool-a-down" and tool_name == "toolA":
        return True
    if mode == "tool-b-down" and tool_name == "toolB":
        return True
    if mode == "tool-a-transient" and tool_name == "toolA":
        return attempt == 1
    if mode == "tool-b-transient" and tool_name == "toolB":
        return attempt == 1
    return False


MAX_RETRIES = 3
RETRY_DELAYS = parse_retry_delays("5,10,20", MAX_RETRIES)
print("MAX_RETRIES =", MAX_RETRIES, "| MAX_ATTEMPTS =", MAX_RETRIES + 1, "| RETRY_DELAYS(sec) =", RETRY_DELAYS)


MAX_RETRIES = 3 | RETRY_DELAYS(sec) = [5, 10, 20]


### 2-2. 도구 실행 함수

> 학습용 노트북에서는 기본적으로 `simulate_wait=False`로 두어 실제로 5/10/20초를 기다리지 않고
> 재시도 로그만 즉시 생성합니다. 실제 대기 시간을 체감하고 싶다면 아래 `SIMULATE_WAIT = True`로 바꿔 실행하세요
> (Q1 기준 both-down 모드는 최대 (5+10)×2 ≈ 30초가 소요됩니다).

In [10]:
SIMULATE_WAIT = False  # True로 바꾸면 재시도 사이에 실제로 sleep합니다.


def execute_tool(
    *,
    tool_name: str,
    payload: Any,
    mode: str,
    max_retries: int,
    retry_delays: list[int],
    simulate_wait: bool,
) -> dict[str, Any]:
    logs: list[dict[str, Any]] = []
    for attempt in range(1, max_retries + 2):
        failed = should_tool_fail(mode, tool_name, attempt)
        if not failed:
            logs.append({"attempt": attempt, "status": "ok"})
            return {
                "status": "ok",
                "attempts": attempt,
                "logs": logs,
                "payload": payload,
            }
        logs.append({"attempt": attempt, "status": "fail", "error": "simulated_tool_failure"})
        if attempt <= max_retries and simulate_wait:
            time.sleep(retry_delays[attempt - 1])
    return {
        "status": "fail",
        "attempts": max_retries + 1,
        "logs": logs,
        "payload": None,
    }

print("execute_tool 정의 완료")


execute_tool 정의 완료


### 2-3. 통합 응답 구성 (fallback 정책: 정상 / 부분(partial) / 차단(blocked))

In [11]:
def compose_response(
    *,
    scenario: dict[str, Any],
    tool_a_result: dict[str, Any],
    tool_b_result: dict[str, Any],
) -> dict[str, Any]:
    warnings: list[str] = []
    evidence_links: list[dict[str, str]] = []
    key_findings: list[str] = []
    actions: list[str] = []
    source_trace: list[dict[str, Any]] = []
    overall_status = "pass"

    tool_a_ok = tool_a_result["status"] == "ok"
    tool_b_ok = tool_b_result["status"] == "ok"

    if tool_a_ok and tool_a_result["payload"]:
        key_findings.extend(tool_a_result["payload"].get("highlights", []))
        source_trace.append({"iq": "FabricIQ", "role": "structured", "origin": "track1-csv-simulation", "semanticKeys": scenario.get("semanticKeys", [])})
    if tool_b_ok and tool_b_result["payload"]:
        evidence_links = tool_b_result["payload"].get("evidence", [])[:5]
        source_trace.append({"iq": "WorkIQ", "role": "unstructured", "origin": "track2-manifest-simulation", "semanticKeys": scenario.get("semanticKeys", [])})

    if not tool_a_ok and not tool_b_ok:
        overall_status = "blocked"
        warnings.append("Tool A/B 모두 실패: 답변 생성을 중단하고 차단 원인 및 복구 조치만 반환합니다.")
        key_findings = ["정형·비정형 도구가 모두 실패해 분석을 지속할 수 없습니다."]
        actions = [
            "권한/토큰 상태를 먼저 복구합니다.",
            "인덱스 범위와 커넥터 상태를 재점검합니다.",
            "복구 후 표준 질문 Q1으로 재시도합니다.",
        ]
        evidence_links = []
    elif not tool_a_ok:
        overall_status = "partial"
        warnings.append("정형 수치 미검증")
        actions = [
            "Tool A(FabricIQ) 인증 또는 SQL endpoint 연결을 복구합니다.",
            "복구 후 동일 질문으로 정형 지표를 재수집합니다.",
        ]
    elif not tool_b_ok:
        overall_status = "partial"
        warnings.append("업무 문서 근거 없음")
        actions = [
            "Tool B(WorkIQ) 권한/인덱스 최신성을 확인합니다.",
            "복구 후 동일 질문으로 근거 링크를 재수집합니다.",
        ]
    else:
        actions = [
            "근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.",
            "응답 품질 점수(정확도/근거성/환각률)를 기록합니다.",
        ]

    response = {
        "question": scenario["question"],
        "overallStatus": overall_status,
        "summary": f"{scenario['id']} 실행 결과: {overall_status}",
        "keyFindings": key_findings,
        "warnings": warnings,
        "recommendedActions": actions,
        "evidenceLinks": evidence_links,
        "sourceTrace": source_trace,
        "qualityChecks": {
            "hasStructuredMetric": tool_a_ok and bool(key_findings),
            "hasEvidenceLink": len(evidence_links) > 0,
            "hasBothSources": tool_a_ok and tool_b_ok and bool(key_findings) and bool(evidence_links),
        },
    }
    return response


def run_scenario(scenario_id: str, mode: str) -> dict[str, Any]:
    scenario = scenario_by_id[scenario_id]
    tool_a_payload = tool_a_metrics.get(scenario_id, {})
    tool_b_payload = tool_b_evidence.get(scenario_id, {})

    tool_a_result = execute_tool(
        tool_name="toolA", payload=tool_a_payload, mode=mode,
        max_retries=MAX_RETRIES, retry_delays=RETRY_DELAYS, simulate_wait=SIMULATE_WAIT,
    )
    tool_b_result = execute_tool(
        tool_name="toolB", payload=tool_b_payload, mode=mode,
        max_retries=MAX_RETRIES, retry_delays=RETRY_DELAYS, simulate_wait=SIMULATE_WAIT,
    )

    payload = {
        "runContext": {
            "scenarioId": scenario_id,
            "executionMode": "simulation",
            "mode": mode,
            "runAt": datetime.now(timezone.utc).isoformat(),
            "retryPolicy": {"maxRetries": MAX_RETRIES, "retryDelaysSec": RETRY_DELAYS},
        },
        "toolStatus": {
            "toolA": {k: v for k, v in tool_a_result.items() if k != "payload"},
            "toolB": {k: v for k, v in tool_b_result.items() if k != "payload"},
        },
        "response": compose_response(scenario=scenario, tool_a_result=tool_a_result, tool_b_result=tool_b_result),
    }
    output_path = RESPONSE_DIR / f"{scenario_id}__{mode}.json"
    write_json(output_path, payload, pretty=True)
    return payload

scenario_by_id = {scenario["id"]: scenario for scenario in SCENARIOS}
print("run_scenario 정의 완료")


run_scenario 정의 완료


### 2-4. 정상(normal) 모드 — Q1, Q2, Q3 전체 실행

In [12]:
normal_results = {}
for scenario_id in sorted(scenario_by_id.keys()):
    result = run_scenario(scenario_id, "normal")
    normal_results[scenario_id] = result
    response = result["response"]
    print(f"{scenario_id}: overallStatus={response['overallStatus']}, warnings={response['warnings']}, evidenceLinks={len(response['evidenceLinks'])}")


Q1: overallStatus=pass, warnings=[], evidenceLinks=5
Q2: overallStatus=pass, warnings=[], evidenceLinks=5
Q3: overallStatus=pass, warnings=[], evidenceLinks=5


### 2-5. Fallback 시나리오 — Q1을 대상으로 도구 장애 3종 실행

- `tool-a-down`: Tool A(FabricIQ) 실패 → **부분(partial)** 응답, "정형 수치 미검증" 경고
- `tool-b-down`: Tool B(WorkIQ) 실패 → **부분(partial)** 응답, "업무 문서 근거 없음" 경고
- `both-down`: 둘 다 실패 → **차단(blocked)** 응답, 근거 링크 없이 복구 조치만 반환


In [13]:
fallback_results = {}
for mode in ["tool-a-down", "tool-b-down", "both-down"]:
    result = run_scenario("Q1", mode)
    fallback_results[mode] = result
    response = result["response"]
    print(f"mode={mode}: overallStatus={response['overallStatus']}, warnings={response['warnings']}")
    print(f"  recommendedActions: {response['recommendedActions']}")


mode=tool-a-down: overallStatus=partial, warnings=['정형 수치 미검증']
  recommendedActions: ['Tool A(FabricIQ) 인증 또는 SQL endpoint 연결을 복구합니다.', '복구 후 동일 질문으로 정형 지표를 재수집합니다.']
mode=tool-b-down: overallStatus=partial, warnings=['업무 문서 근거 없음']
  recommendedActions: ['Tool B(WorkIQ) 권한/인덱스 최신성을 확인합니다.', '복구 후 동일 질문으로 근거 링크를 재수집합니다.']
mode=both-down: overallStatus=blocked, warnings=['Tool A/B 모두 실패: 답변 생성을 중단하고 차단 원인 및 복구 조치만 반환합니다.']
  recommendedActions: ['권한/토큰 상태를 먼저 복구합니다.', '인덱스 범위와 커넥터 상태를 재점검합니다.', '복구 후 표준 질문 Q1으로 재시도합니다.']


### 2-6. 질문에 대한 실제 답변을 Markdown으로 렌더링

지금까지 실행한 `normal_results`(Q1~Q3)와 `fallback_results`(Q1, 도구 장애 3종)의 `response` 객체가 질문에 대한 실제 답변입니다. 이를 사람이 읽기 쉬운 Markdown으로 변환해 `generated/reports/track3_answers.md`에 저장하고 바로 렌더링해 확인합니다.

In [14]:
def render_answer_markdown(payload: dict[str, Any]) -> str:
    """Render a single Track3 response payload (runContext + response) as a Markdown section."""
    run_context = payload.get("runContext", {})
    response = payload.get("response", {})
    scenario_id = run_context.get("scenarioId", "?")
    mode = run_context.get("mode", "?")

    lines = [f"## {scenario_id} (mode={mode})", ""]
    lines.append(f"**질문:** {response.get('question', '-')}")
    lines.append("")
    lines.append(f"**상태:** {response.get('overallStatus', '-')}")
    lines.append("")
    lines.append(f"**요약:** {response.get('summary', '-')}")
    lines.append("")

    lines.append("### 핵심 발견 (keyFindings)")
    findings = response.get("keyFindings") or []
    if findings:
        lines.extend(f"- {item}" for item in findings)
    else:
        lines.append("- (없음)")
    lines.append("")

    warnings = response.get("warnings") or []
    if warnings:
        lines.append("### 경고 (warnings)")
        lines.extend(f"- ⚠️ {item}" for item in warnings)
        lines.append("")

    actions = response.get("recommendedActions") or []
    if actions:
        lines.append("### 권장 조치 (recommendedActions)")
        lines.extend(f"- {item}" for item in actions)
        lines.append("")

    trace = response.get("sourceTrace") or []
    lines.append("### 3-IQ 소스 추적 (sourceTrace)")
    if trace:
        lines.append("| iq | role | origin | semanticKeys |")
        lines.append("| --- | --- | --- | --- |")
        for item in trace:
            lines.append(f"| {item.get('iq', '-')} | {item.get('role', '-')} | {item.get('origin', '-')} | {', '.join(item.get('semanticKeys', []))} |")
    else:
        lines.append("- (없음)")
    lines.append("")

    links = response.get("evidenceLinks") or []
    lines.append("### 근거 링크 (evidenceLinks)")
    if links:
        lines.append("| source | title | businessDate | reference |")
        lines.append("| --- | --- | --- | --- |")
        for link in links:
            reference = link.get('url') or link.get('location') or link.get('target') or '-'
            lines.append(f"| {link.get('source', '-')} | {link.get('title', '-')} | {link.get('businessDate', '-')} | {reference} |")
    else:
        lines.append("- (없음)")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


ANSWERS_MARKDOWN_PATH = OUTPUT_DIR / "reports" / "track3_answers.md"

all_answer_payloads = list(normal_results.values()) + list(fallback_results.values())
sections = ["# Track3 질문별 실제 답변", ""]
for payload in all_answer_payloads:
    sections.append(render_answer_markdown(payload))

ANSWERS_MARKDOWN_PATH.parent.mkdir(parents=True, exist_ok=True)
ANSWERS_MARKDOWN_PATH.write_text("\n".join(sections), encoding="utf-8")
print("답변 Markdown 저장 완료:", ANSWERS_MARKDOWN_PATH)


답변 Markdown 저장 완료: /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/track3_answers.md


In [15]:
from IPython.display import Markdown, display

display(Markdown(ANSWERS_MARKDOWN_PATH.read_text(encoding='utf-8')))

# Track3 질문별 실제 답변

## Q1 (mode=normal)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** pass

**요약:** Q1 실행 결과: pass

### 핵심 발견 (keyFindings)
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| OneDrive | 캠페인 주간 성과 리뷰 노트 | 2026-05-21 |
| SharePoint | SummerPush 캠페인 킥오프 기획서 | 2026-04-15 |

---

## Q2 (mode=normal)

**질문:** 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?

**상태:** pass

**요약:** Q2 실행 결과: pass

### 핵심 발견 (keyFindings)
- 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | 주문 취소율과 배송 지연 상관 점검 | 2026-05-24T14:00:00+09:00 |
| SharePoint | 배송 지연 원인 분석 및 고객 영향 | 2026-05-23 |
| Outlook | 반품 사유 월간 요약 - 채널 및 고객등급 검토 | 2026-05-25T16:30:00+09:00 |
| OneDrive | 반품 VOC 분류 워크숍 노트 | 2026-05-25 |
| Outlook | RE: [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-17T08:50:00+09:00 |

---

## Q3 (mode=normal)

**질문:** Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?

**상태:** pass

**요약:** Q3 실행 결과: pass

### 핵심 발견 (keyFindings)
- Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.
- 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| SharePoint | Q3 리더십 운영 리스크 브리핑 | 2026-07-11 |
| Outlook | RE: BackToSchool 캠페인 조건부 승인 요청 | 2026-07-11T09:30:00+09:00 |
| OneDrive | 재고·물류·CS 합동 회의록 | 2026-05-17 |
| Teams | 핵심 상품 품절 임박 공동 대응 | 2026-05-16T09:00:00+09:00 |
| Outlook | [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-16T09:15:00+09:00 |

---

## Q1 (mode=tool-a-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** partial

**요약:** Q1 실행 결과: partial

### 핵심 발견 (keyFindings)
- (없음)

### 경고 (warnings)
- ⚠️ 정형 수치 미검증

### 권장 조치 (recommendedActions)
- Tool A(FabricIQ) 인증 또는 SQL endpoint 연결을 복구합니다.
- 복구 후 동일 질문으로 정형 지표를 재수집합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| OneDrive | 캠페인 주간 성과 리뷰 노트 | 2026-05-21 |
| SharePoint | SummerPush 캠페인 킥오프 기획서 | 2026-04-15 |

---

## Q1 (mode=tool-b-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** partial

**요약:** Q1 실행 결과: partial

### 핵심 발견 (keyFindings)
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.

### 경고 (warnings)
- ⚠️ 업무 문서 근거 없음

### 권장 조치 (recommendedActions)
- Tool B(WorkIQ) 권한/인덱스 최신성을 확인합니다.
- 복구 후 동일 질문으로 근거 링크를 재수집합니다.

### 근거 링크 (evidenceLinks)
- (없음)

---

## Q1 (mode=both-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** blocked

**요약:** Q1 실행 결과: blocked

### 핵심 발견 (keyFindings)
- 정형·비정형 도구가 모두 실패해 분석을 지속할 수 없습니다.

### 경고 (warnings)
- ⚠️ Tool A/B 모두 실패: 답변 생성을 중단하고 차단 원인 및 복구 조치만 반환합니다.

### 권장 조치 (recommendedActions)
- 권한/토큰 상태를 먼저 복구합니다.
- 인덱스 범위와 커넥터 상태를 재점검합니다.
- 복구 후 표준 질문 Q1으로 재시도합니다.

### 근거 링크 (evidenceLinks)
- (없음)

---


---
# 3부. 품질 게이트 평가 (`evaluate_track3_outputs.py` 로직)

생성된 응답 파일들을 모드별 기대 정책과 비교해 pass/fail을 판정합니다.


### 3-1. 응답별 판정 규칙

In [16]:
def contains_warning(warnings: list[str], token: str) -> bool:
    normalized = token.strip().lower()
    return any(normalized in warning.strip().lower() for warning in warnings)


def evaluate_response(payload: dict[str, Any]) -> tuple[bool, list[str]]:
    run_context = payload.get("runContext", {})
    response = payload.get("response", {})
    mode = run_context.get("mode", "")
    retry_policy = run_context.get("retryPolicy", {})
    tool_status = payload.get("toolStatus", {})

    overall_status = response.get("overallStatus", "")
    warnings = response.get("warnings", [])
    links = response.get("evidenceLinks", [])
    source_trace = response.get("sourceTrace", [])
    quality_checks = response.get("qualityChecks", {})
    traced_iq = {item.get("iq") for item in source_trace if isinstance(item, dict)}

    reasons: list[str] = []
    passed = True

    if retry_policy.get("maxRetries") != 3 or retry_policy.get("retryDelaysSec") != [5, 10, 20]:
        passed = False
        reasons.append("retry policy must be maxRetries=3 with delays [5, 10, 20]")

    if mode in {"normal", "tool-a-transient", "tool-b-transient"}:
        if overall_status != "pass":
            passed = False
            reasons.append(f"expected overallStatus=pass but got {overall_status}")
        if not quality_checks.get("hasStructuredMetric", False):
            passed = False
            reasons.append("structured metric missing")
        if len(links) < 2:
            passed = False
            reasons.append("evidence links < 2")
        if warnings:
            passed = False
            reasons.append("warnings should be empty in pass mode")
        if traced_iq != {"FabricIQ", "WorkIQ"}:
            passed = False
            reasons.append("sourceTrace must identify FabricIQ and WorkIQ")
    elif mode == "tool-a-down":
        if overall_status != "partial":
            passed = False
            reasons.append(f"expected partial for tool-a-down but got {overall_status}")
        if not contains_warning(warnings, "정형 수치 미검증"):
            passed = False
            reasons.append("missing warning: 정형 수치 미검증")
        if tool_status.get("toolA", {}).get("attempts") != 4:
            passed = False
            reasons.append("tool-a-down must record initial call plus 3 retries")
    elif mode == "tool-b-down":
        if overall_status != "partial":
            passed = False
            reasons.append(f"expected partial for tool-b-down but got {overall_status}")
        if not contains_warning(warnings, "업무 문서 근거 없음"):
            passed = False
            reasons.append("missing warning: 업무 문서 근거 없음")
        if tool_status.get("toolB", {}).get("attempts") != 4:
            passed = False
            reasons.append("tool-b-down must record initial call plus 3 retries")
    elif mode == "both-down":
        if overall_status != "blocked":
            passed = False
            reasons.append(f"expected blocked for both-down but got {overall_status}")
        if links:
            passed = False
            reasons.append("both-down must not return evidence links")
        if tool_status.get("toolA", {}).get("attempts") != 4 or tool_status.get("toolB", {}).get("attempts") != 4:
            passed = False
            reasons.append("both-down must record initial call plus 3 retries per tool")
    else:
        passed = False
        reasons.append(f"unknown mode: {mode}")

    return passed, reasons

print("evaluate_response 정의 완료")


evaluate_response 정의 완료


### 3-2. `generated/responses/*.json` 전체를 평가하고 리포트 저장

In [17]:
files = sorted(RESPONSE_DIR.glob("*.json"))
if not files:
    raise RuntimeError(f"No response files found in: {RESPONSE_DIR}")

results: list[dict[str, Any]] = []
failed_count = 0

for file_path in files:
    payload = json.loads(file_path.read_text(encoding="utf-8"))
    passed, reasons = evaluate_response(payload)
    if not passed:
        failed_count += 1
    results.append(
        {
            "file": str(file_path),
            "scenarioId": payload.get("runContext", {}).get("scenarioId"),
            "mode": payload.get("runContext", {}).get("mode"),
            "passed": passed,
            "reasons": reasons,
        }
    )

report = {
    "responsesDir": str(RESPONSE_DIR),
    "total": len(results),
    "passed": len(results) - failed_count,
    "failed": failed_count,
    "results": results,
}

write_json(REPORT_PATH, report, pretty=True)


def render_markdown_report(report: dict[str, Any]) -> str:
    """Render the evaluation JSON report as a human-readable Markdown document."""
    lines: list[str] = []
    lines.append("# Track3 Evaluation Report")
    lines.append("")
    lines.append(f"- responsesDir: `{report['responsesDir']}`")
    lines.append(f"- total: {report['total']}")
    lines.append(f"- passed: {report['passed']}")
    lines.append(f"- failed: {report['failed']}")
    lines.append("")
    lines.append("## Results")
    lines.append("")
    lines.append("| Scenario | Mode | Result | Reasons |")
    lines.append("| --- | --- | --- | --- |")
    for row in report["results"]:
        status = "✅ PASS" if row["passed"] else "❌ FAIL"
        reasons = "; ".join(row["reasons"]) if row["reasons"] else "-"
        lines.append(f"| {row['scenarioId']} | {row['mode']} | {status} | {reasons} |")
    lines.append("")
    return "\n".join(lines)


# JSON 리포트를 만든 직후, 동일 내용을 Markdown 리포트로도 저장합니다.
MARKDOWN_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
MARKDOWN_REPORT_PATH.write_text(render_markdown_report(report), encoding="utf-8")

print("[Track3 Evaluation]")
print(f"- total: {report['total']}")
print(f"- passed: {report['passed']}")
print(f"- failed: {report['failed']}")
print(f"- report (json): {REPORT_PATH}")
print(f"- report (markdown): {MARKDOWN_REPORT_PATH}")
for row in results:
    status = "PASS" if row["passed"] else "FAIL"
    print(f"  [{status}] {row['scenarioId']}__{row['mode']}" + (f" -> {row['reasons']}" if row["reasons"] else ""))


[Track3 Evaluation]
- total: 8
- passed: 8
- failed: 0
- report (json): /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/evaluation_report.json
- report (markdown): /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/evaluation_report.md
  [PASS] Q1__both-down
  [PASS] Q1__normal
  [PASS] Q1__tool-a-down
  [PASS] Q1__tool-b-down
  [PASS] Q2__normal
  [PASS] Q2__tool-a-transient
  [PASS] Q3__normal
  [PASS] Q3__tool-b-transient


### 3-2-1. Markdown 리포트 렌더링 확인

방금 저장된 `evaluation_report.md`를 노트북에서 바로 렌더링해 확인합니다.

In [18]:
from IPython.display import Markdown, display

display(Markdown(MARKDOWN_REPORT_PATH.read_text(encoding='utf-8')))

# Track3 Evaluation Report

- responsesDir: `/Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/responses`
- total: 8
- passed: 8
- failed: 0

## Results

| Scenario | Mode | Result | Reasons |
| --- | --- | --- | --- |
| Q1 | both-down | ✅ PASS | - |
| Q1 | normal | ✅ PASS | - |
| Q1 | tool-a-down | ✅ PASS | - |
| Q1 | tool-b-down | ✅ PASS | - |
| Q2 | normal | ✅ PASS | - |
| Q2 | tool-a-transient | ✅ PASS | - |
| Q3 | normal | ✅ PASS | - |
| Q3 | tool-b-transient | ✅ PASS | - |


### 3-3. (선택) `--strict`처럼 실패가 있으면 예외를 발생시켜 확인

CI/자동화 환경에서는 `evaluate_track3_outputs.py --strict`처럼 실패 시 프로세스를 중단시킵니다.
아래 셀을 실행해 학습용으로 동일하게 확인해볼 수 있습니다 (실패가 없으면 통과 메시지만 출력).

In [19]:
if failed_count > 0:
    raise SystemExit(f"[strict] {failed_count}건 실패 — 위 결과의 reasons를 확인하세요.")
else:
    print("[strict] 모든 응답이 품질 게이트를 통과했습니다.")


[strict] 모든 응답이 품질 게이트를 통과했습니다.


---
# 4부. 임원용 리더십 브리핑 초안 생성

[track3/WORKBOOK.md](../WORKBOOK.md)의 **미션 4(생성/평가 + 제출)**에서는 참가자가 FoundryIQ 에이전트에게
"오늘 아침 리더십 브리핑 형식으로 요약해줘"처럼 요청해 아래 고정 출력 형식(`[TRACK3_RESPONSE]`)의 브리핑을 만들고,
Q1~Q3를 하나로 묶은 **최종 브리핑 1건**을 제출합니다.

여기서는 실제 LLM 생성 없이, 지금까지 만든 Q1~Q3 정상(normal) 응답을 **규칙 기반으로 결합**해 임원이 바로 읽을 수 있는
1페이지 브리핑 초안(Executive Summary + 수치근거 + 문서근거 + 조치안 + 주의사항)을 자동으로 만들어봅니다.

> 이 초안은 출발점입니다. 실제 제출 전에는 FoundryIQ 에이전트/사람이 문장을 다듬고 팀 판단을 더해 완성해야 합니다.


In [20]:
def render_track3_response_block(payload: dict[str, Any]) -> str:
    """WORKBOOK.md의 [TRACK3_RESPONSE] 고정 출력 형식으로 단일 시나리오 응답을 렌더링합니다."""
    response = payload.get("response", {})
    metrics = response.get("keyFindings") or ["-"]
    links = response.get("evidenceLinks") or []
    link_summary = "; ".join(f"{l.get('source', '-')}:{l.get('title', '-')}" for l in links) or "-"
    source_trace = response.get("sourceTrace") or []
    trace_summary = "; ".join(f"{item.get('iq', '-')}:{item.get('role', '-')}" for item in source_trace) or "-"
    actions = response.get("recommendedActions") or ["-"]
    warnings = response.get("warnings") or []

    lines = [
        "[TRACK3_RESPONSE]",
        f"question={response.get('question', '-')}",
        f"summary={response.get('summary', '-')}",
        f"structuredMetrics={' / '.join(metrics)}",
        f"evidenceLinks={link_summary}",
        f"sourceTrace={trace_summary}",
        f"actions={' / '.join(actions)}",
        f"warnings={'; '.join(warnings) if warnings else '없음'}",
        "[/TRACK3_RESPONSE]",
    ]
    return "\n".join(lines)


def compose_leadership_briefing(scenario_payloads: dict[str, dict[str, Any]]) -> str:
    """Q1~Q3 정상 응답을 결합해 임원용 리더십 브리핑 초안(Markdown)을 생성합니다."""
    generated_at = datetime.now(timezone.utc).isoformat()
    lines: list[str] = []
    lines.append("# Track3 리더십 브리핑 (자동 초안)")
    lines.append("")
    lines.append(f"- 생성일시(UTC): {generated_at}")
    lines.append("- 대상 시나리오: " + ", ".join(sorted(scenario_payloads.keys())))
    lines.append("")

    lines.append("## 한눈에 보기 (Executive Summary)")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        first_finding = (response.get("keyFindings") or ["-"])[0]
        lines.append(f"- **{scenario_id}** — {response.get('question', '-')}: {first_finding}")
    lines.append("")

    lines.append("## 핵심 수치 근거 (Structured Metrics)")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        lines.append(f"### {scenario_id}")
        for finding in response.get("keyFindings") or ["-"]:
            lines.append(f"- {finding}")
    lines.append("")

    lines.append("## 문서 근거 (Evidence Links, 시나리오별 상위 3건)")
    lines.append("| Scenario | source | title | businessDate |")
    lines.append("| --- | --- | --- | --- |")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for link in (response.get("evidenceLinks") or [])[:3]:
            lines.append(f"| {scenario_id} | {link.get('source', '-')} | {link.get('title', '-')} | {link.get('businessDate', '-')} |")
    lines.append("")

    lines.append("## 즉시 조치 제안 (Actions)")
    seen_actions: set[str] = set()
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for action in response.get("recommendedActions") or []:
            if action not in seen_actions:
                lines.append(f"- {action}")
                seen_actions.add(action)
    lines.append("")

    lines.append("## 주의사항 (Warnings)")
    any_warning = False
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for warning in response.get("warnings") or []:
            lines.append(f"- **{scenario_id}**: ⚠️ {warning}")
            any_warning = True
    if not any_warning:
        lines.append("- 없음 (Q1~Q3 모두 정형 수치 + 근거 링크 정상 확보)")
    lines.append("")

    lines.append("## Appendix — 시나리오별 [TRACK3_RESPONSE] 제출 블록")
    lines.append("")
    lines.append("```text")
    for scenario_id in sorted(scenario_payloads.keys()):
        lines.append(render_track3_response_block(scenario_payloads[scenario_id]))
        lines.append("")
    lines.append("```")
    lines.append("")

    return "\n".join(lines)

print("compose_leadership_briefing 정의 완료")


compose_leadership_briefing 정의 완료


In [21]:
BRIEFING_PATH = OUTPUT_DIR / "reports" / "leadership_briefing.md"

briefing_markdown = compose_leadership_briefing(normal_results)
BRIEFING_PATH.parent.mkdir(parents=True, exist_ok=True)
BRIEFING_PATH.write_text(briefing_markdown, encoding="utf-8")
print("리더십 브리핑 초안 저장 완료:", BRIEFING_PATH)


리더십 브리핑 초안 저장 완료: /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/leadership_briefing.md


In [22]:
from IPython.display import Markdown, display

display(Markdown(BRIEFING_PATH.read_text(encoding='utf-8')))

# Track3 리더십 브리핑 (자동 초안)

- 생성일시(UTC): 2026-07-13T18:13:07.635844+00:00
- 대상 시나리오: Q1, Q2, Q3

## 한눈에 보기 (Executive Summary)
- **Q1** — 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?: 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- **Q2** — 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?: 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- **Q3** — Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?: Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.

## 핵심 수치 근거 (Structured Metrics)
### Q1
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.
### Q2
- 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.
### Q3
- Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.
- 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.

## 문서 근거 (Evidence Links, 시나리오별 상위 3건)
| Scenario | source | title | businessDate |
| --- | --- | --- | --- |
| Q1 | Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| Q1 | SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Q1 | Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| Q2 | Teams | 주문 취소율과 배송 지연 상관 점검 | 2026-05-24T14:00:00+09:00 |
| Q2 | SharePoint | 배송 지연 원인 분석 및 고객 영향 | 2026-05-23 |
| Q2 | Outlook | 반품 사유 월간 요약 - 채널 및 고객등급 검토 | 2026-05-25T16:30:00+09:00 |
| Q3 | SharePoint | Q3 리더십 운영 리스크 브리핑 | 2026-07-11 |
| Q3 | Outlook | RE: BackToSchool 캠페인 조건부 승인 요청 | 2026-07-11T09:30:00+09:00 |
| Q3 | OneDrive | 재고·물류·CS 합동 회의록 | 2026-05-17 |

## 즉시 조치 제안 (Actions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

## 주의사항 (Warnings)
- 없음 (Q1~Q3 모두 정형 수치 + 근거 링크 정상 확보)

## Appendix — 시나리오별 [TRACK3_RESPONSE] 제출 블록

```text
[TRACK3_RESPONSE]
question=결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?
summary=Q1 실행 결과: pass
structuredMetrics=핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다. / payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.
evidenceLinks=Teams:SummerPush 중간 성과 해석; SharePoint:SummerPush 중간 성과 리포트; Outlook:[리더십] 5월 매출 급락 이슈 공유; OneDrive:캠페인 주간 성과 리뷰 노트; SharePoint:SummerPush 캠페인 킥오프 기획서
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

[TRACK3_RESPONSE]
question=배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?
summary=Q2 실행 결과: pass
structuredMetrics=배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다. / 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.
evidenceLinks=Teams:주문 취소율과 배송 지연 상관 점검; SharePoint:배송 지연 원인 분석 및 고객 영향; Outlook:반품 사유 월간 요약 - 채널 및 고객등급 검토; OneDrive:반품 VOC 분류 워크숍 노트; Outlook:RE: [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

[TRACK3_RESPONSE]
question=Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?
summary=Q3 실행 결과: pass
structuredMetrics=Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다. / 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.
evidenceLinks=SharePoint:Q3 리더십 운영 리스크 브리핑; Outlook:RE: BackToSchool 캠페인 조건부 승인 요청; OneDrive:재고·물류·CS 합동 회의록; Teams:핵심 상품 품절 임박 공동 대응; Outlook:[긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

```


---
# 5부. FoundryIQ 에이전트(LLM)로 최종 리더십 브리핑 문서 생성

4부에서 만든 `leadership_briefing.md`는 **규칙 기반 초안**입니다. 이 섹션은 실제 **Azure AI Foundry Responses API**를
호출해, 그 초안을 [track3/WORKBOOK.md](../WORKBOOK.md) 미션 2의 시스템 프롬프트 정책(정형 우선 + 근거 결합,
근거 없는 단정 금지, `핵심요약/수치근거/문서근거/조치안/주의사항` 출력 형식)에 맞춰 임원이 바로 읽을 수 있는
완성된 문장으로 다듬습니다.

## 사전 준비 (환경변수)

Azure AI Foundry에서 Responses API를 지원하는 모델을 배포한 뒤, 아래 환경변수를 설정하세요. 노트북을 열기 전에
터미널에서 `export`하면 커널이 값을 읽을 수 있습니다.

| 환경변수 | 설명 | 필수 여부 |
|---|---|---|
| `AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT` | Responses API 전체 엔드포인트 (예: `https://<resource>.services.ai.azure.com/openai/v1/responses`) | 필수 |
| `AZURE_AI_FOUNDRY_MODEL` | Responses API 요청의 `model` 값 | 필수 |
| `AZURE_AI_FOUNDRY_API_KEY` | 키 인증 사용 시 `api-key` 헤더 값 | 조건부 |
| `AZURE_AI_FOUNDRY_BEARER_TOKEN` | Entra 액세스 토큰 사용 시 `Authorization: Bearer` 값 | 조건부 |

> API key와 Bearer token은 서로 다른 인증 값입니다. JWT 형식 액세스 토큰을 API key 변수에 넣지 마세요. 인증 값이
> 없어도 나머지 셀은 정상 동작하며, 이 섹션은 **자동으로 건너뛰고** 4부의 규칙 기반 초안을 유지합니다.


In [ ]:
import sys

if str(TRACK3_ROOT) not in sys.path:
    sys.path.insert(0, str(TRACK3_ROOT))

from foundry_responses import FoundryResponsesConfig, generate_leadership_briefing

print("Foundry Responses API 공용 모듈 로드 완료")


In [ ]:
# 연결 정보는 환경변수에서만 읽으며 노트북 출력에는 비밀 값을 표시하지 않습니다.
foundry_config = FoundryResponsesConfig.from_env()
FOUNDRY_CONFIGURED = foundry_config.is_configured

print("AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT =", foundry_config.endpoint or "(미설정)")
print("AZURE_AI_FOUNDRY_MODEL              =", foundry_config.model or "(미설정)")
print("인증 방식                            =", foundry_config.auth_mode)
print("FOUNDRY_CONFIGURED                  =", FOUNDRY_CONFIGURED)


In [ ]:
def generate_llm_leadership_briefing(draft_markdown: str) -> str:
    """규칙 기반 초안을 Foundry Responses API로 다듬어 최종 브리핑을 생성합니다."""
    return generate_leadership_briefing(draft_markdown, config=foundry_config)


print("generate_llm_leadership_briefing 정의 완료")


In [ ]:
LLM_BRIEFING_PATH = OUTPUT_DIR / "reports" / "leadership_briefing_llm.md"

if not FOUNDRY_CONFIGURED:
    print("[건너뜀] Foundry Responses API 환경변수가 설정되지 않아 LLM 기반 최종본 생성을 건너뜁니다.")
    print("필수: AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT, AZURE_AI_FOUNDRY_MODEL, 인증(API_KEY 또는 BEARER_TOKEN)")
    print("규칙 기반 초안(generated/reports/leadership_briefing.md)을 그대로 제출용으로 사용할 수 있습니다.")
else:
    draft_markdown = BRIEFING_PATH.read_text(encoding="utf-8")
    llm_output = generate_llm_leadership_briefing(draft_markdown)
    LLM_BRIEFING_PATH.parent.mkdir(parents=True, exist_ok=True)
    LLM_BRIEFING_PATH.write_text(llm_output, encoding="utf-8")
    print("Foundry LLM 최종 브리핑 저장 완료:", LLM_BRIEFING_PATH)

    from IPython.display import Markdown, display
    display(Markdown(llm_output))


---
# 마무리

이 노트북 한 곳에서 아래 흐름을 모두 실행했습니다.

1. **샘플 생성**: Track1 CSV + Track2 매니페스트 → `scenarios.json`, `tool_a_metrics.json`, `tool_b_evidence.json`
2. **통합 응답 시뮬레이션**: 정상 모드(Q1~Q3) + fallback 모드(`tool-a-down`, `tool-b-down`, `both-down`, Q1 기준)
3. **품질 게이트 평가**: `generated/reports/evaluation_report.json`(원본) + `generated/reports/evaluation_report.md`(사람이 읽기 쉬운 Markdown)으로 pass/fail 판정
4. **임원용 리더십 브리핑 초안**: Q1~Q3 응답을 결합한 `generated/reports/leadership_briefing.md` 생성
5. **FoundryIQ LLM 최종본(선택)**: Azure AI Foundry에 배포된 LLM으로 초안을 다듬은 `generated/reports/leadership_briefing_llm.md` 생성 (환경변수 미설정 시 자동 건너뜀)

## 다음 단계

- 결과 파일(응답 JSON, 답변 Markdown `track3_answers.md`, 평가 리포트)은 `track3/data/generated/`에 저장되어 있으니, [Track3_Mission_Workbench.ipynb](./Track3_Mission_Workbench.ipynb)나
  CLI(`python evaluate_track3_outputs.py --strict`)로 동일하게 검증해볼 수 있습니다.
- 실습 중 값을 바꿔보고 싶다면: `SCENARIOS`에 새 질문 추가, `CORE_PRODUCTS`/`CORE_CAMPAIGNS` 범위 조정,
  `MAX_RETRIES`/`RETRY_DELAYS` 정책 변경 등을 시도해보세요.
- 다음 문서를 함께 참고하세요: [track3/WORKBOOK.md](../WORKBOOK.md), [track3/docs/Track3_FoundryIQ_Introduction_and_Technical_Guide.md](../docs/Track3_FoundryIQ_Introduction_and_Technical_Guide.md)
